In [ ]:
!pip install torch torchvision -q

In [5]:
!python dataset.py


Dataset
  Entities (N):                    1000
  Relations (R):                   11  →  N×R = 11000
  Extraction relations:            ['class', 'color', 'material', 'origin', 'shape', 'size']
  Composition relations:           ['same_color_as', 'same_shape_as', 'same_size_as', 'same_material_as', 'same_origin_as']
  Entities with comp in training:  800  (80%)
  Held-out for generalization:     200  (20%)
  Training queries:                10000
  Test queries (N×R):              11000

Sample entity:
  {
    "name": "Bliblax-210",
    "class": "warrior",
    "color": "green",
    "material": "wood",
    "origin": "region_B",
    "shape": "star",
    "size": "huge",
    "same_color_as": "Glonvix-314",
    "same_shape_as": "Crinnik-342",
    "same_size_as": "Splewynn-991",
    "same_material_as": "Faxvix-86",
    "same_origin_as": "Glonvix-314"
}

Training sequences for Bliblax-210:
  [extraction] <S> Bliblax-210 </S> <R> class </R> <O> warrior </O>
  [extraction] <S> Bliblax-210 </S

In [6]:
!python tokenizer.py

  Loaded 1000 entity names from dataset.json
SROTokenizer built:
  Special tokens:   9
  Attribute values: 35
  Relation words:   8
  Entity names:     1000  (from dataset.json)
  Total vocab:      1052
  Tokenizer saved → tokenizer.json

── Encoding examples ──

  Input:   <S> Bliblax-210 </S> <R> color </R> <O> blue </O>
  Tokens:  ['<S>', 'Bliblax-210', '</S>', '<R>', 'color', '</R>', '<O>', 'blue', '</O>']
  IDs:     [2, 52, 3, 4, 46, 5, 6, 10, 7]
  Decoded: <S> Bliblax-210 </S> <R> color </R> <O> blue </O>

  Input:   <S> Bliforn-232 </S> <R> same color as </R> <O> Blimpnik-34 </O>
  Tokens:  ['<S>', 'Bliforn-232', '</S>', '<R>', 'same', 'color', 'as', '</R>', '<O>', 'Blimpnik-34', '</O>']
  IDs:     [2, 57, 3, 4, 49, 46, 44, 5, 6, 1, 7]
  Decoded: <S> Bliforn-232 </S> <R> same color as </R> <O> <UNK> </O>

  Input:   <S> Bligast-300 </S> <R> material </R> <O> metal </O>
  Tokens:  ['<S>', 'Bligast-300', '</S>', '<R>', 'material', '</R>', '<O>', 'metal', '</O>']
  IDs:     [2, 59,

In [7]:
!python train_experiment.py --model both --out ./outputs

Device: cuda
  Tokenizer loaded from tokenizer.json  (vocab size: 1052)

Building dataloaders...
  Train sequences: 15187  (475 batches)
  Val   sequences: 1687  (53 batches)

Training: SUMMED (baseline)
  [summed] 3,730,944 parameters  | 4 blocks | 4 heads
  [summed] early stopping: patience=5, eval_every=200, min_steps=1000
  [summed] step    200 | loss 3.3183 | lr 3.98e-05
  [summed] step    200 | val_ppl 25.6900  ✓ best
  [summed] step    400 | loss 1.3082 | lr 7.98e-05
  [summed] step    400 | val_ppl 3.6214  ✓ best
  [summed] step    600 | loss 0.8171 | lr 1.00e-04
  [summed] step    600 | val_ppl 2.2747  ✓ best
  [summed] step    800 | loss 0.5629 | lr 1.00e-04
  [summed] step    800 | val_ppl 1.6938  ✓ best
  [summed] step   1000 | loss 0.3480 | lr 9.99e-05
  [summed] step   1000 | val_ppl 1.3997  ✓ best
  [summed] step   1200 | loss 0.2232 | lr 9.99e-05
  [summed] step   1200 | val_ppl 1.2465  ✓ best
  [summed] step   1400 | loss 0.1724 | lr 9.98e-05
  [summed] step   1400 | v

In [8]:
!python evaluate.py \
--summed outputs/summed/model_best.pt \
--disentangled outputs/disentangled/model_best.pt \
--dataset dataset.json \
--out eval_results.json

Loading dataset...
  Loaded 1000 entity names from dataset.json
SROTokenizer built:
  Special tokens:   9
  Attribute values: 35
  Relation words:   8
  Entity names:     1000  (from dataset.json)
  Total vocab:      1052
Building valid answer sets for composition queries...

  Valid answer counts for 'Bliblax-210':
    same_color_as          → 128 valid answers  (attr=green)
    same_shape_as          → 140 valid answers  (attr=star)
    same_size_as           → 216 valid answers  (attr=huge)
    same_material_as       → 154 valid answers  (attr=wood)
    same_origin_as         → 183 valid answers  (attr=region_B)

Loading models...
  [summed] 3,730,944 parameters  | 4 blocks | 4 heads
  [disentangled] 3,639,040 parameters  | 4 blocks | 4 heads

──────────────────────────────────────────────────
Evaluating: summed
──────────────────────────────────────────────────
① N×R accuracy...
② Compositional generalization...

════════════════════════════════════════════════════════════
 summed


In [13]:
import torch
ckpt = torch.load("outputs/summed/summed_model_best2.pt", map_location="cpu")
for k, v in ckpt.items():
    print(k, v.shape)

embed.token_emb.weight torch.Size([1052, 256])
embed.pos_emb.weight torch.Size([128, 256])
blocks.0.ln1.weight torch.Size([256])
blocks.0.ln1.bias torch.Size([256])
blocks.0.ln2.weight torch.Size([256])
blocks.0.ln2.bias torch.Size([256])
blocks.0.attn.in_proj_weight torch.Size([768, 256])
blocks.0.attn.in_proj_bias torch.Size([768])
blocks.0.attn.out_proj.weight torch.Size([256, 256])
blocks.0.attn.out_proj.bias torch.Size([256])
blocks.0.fc1.weight torch.Size([1024, 256])
blocks.0.fc1.bias torch.Size([1024])
blocks.0.fc2.weight torch.Size([256, 1024])
blocks.0.fc2.bias torch.Size([256])
blocks.1.ln1.weight torch.Size([256])
blocks.1.ln1.bias torch.Size([256])
blocks.1.ln2.weight torch.Size([256])
blocks.1.ln2.bias torch.Size([256])
blocks.1.attn.in_proj_weight torch.Size([768, 256])
blocks.1.attn.in_proj_bias torch.Size([768])
blocks.1.attn.out_proj.weight torch.Size([256, 256])
blocks.1.attn.out_proj.bias torch.Size([256])
blocks.1.fc1.weight torch.Size([1024, 256])
blocks.1.fc1.bia

In [15]:
import torch
ckpt = torch.load("outputs/disentangled/concatenated_model_best2.pt", map_location="cpu")
for k, v in ckpt.items():
    print(k, v.shape)

embed.token_emb.weight torch.Size([1052, 192])
embed.pos_emb.weight torch.Size([128, 64])
blocks.0.ln1.weight torch.Size([256])
blocks.0.ln1.bias torch.Size([256])
blocks.0.ln2.weight torch.Size([256])
blocks.0.ln2.bias torch.Size([256])
blocks.0.attn.in_proj_weight torch.Size([768, 256])
blocks.0.attn.in_proj_bias torch.Size([768])
blocks.0.attn.out_proj.weight torch.Size([256, 256])
blocks.0.attn.out_proj.bias torch.Size([256])
blocks.0.fc1.weight torch.Size([1024, 256])
blocks.0.fc1.bias torch.Size([1024])
blocks.0.fc2.weight torch.Size([256, 1024])
blocks.0.fc2.bias torch.Size([256])
blocks.1.ln1.weight torch.Size([256])
blocks.1.ln1.bias torch.Size([256])
blocks.1.ln2.weight torch.Size([256])
blocks.1.ln2.bias torch.Size([256])
blocks.1.attn.in_proj_weight torch.Size([768, 256])
blocks.1.attn.in_proj_bias torch.Size([768])
blocks.1.attn.out_proj.weight torch.Size([256, 256])
blocks.1.attn.out_proj.bias torch.Size([256])
blocks.1.fc1.weight torch.Size([1024, 256])
blocks.1.fc1.bias

In [16]:
from dataset import load_dataset, COMPOSITION_ATTRS
from tokenizer import SROTokenizer
from train import Config, load_model
from evaluate import build_valid_answers
import torch
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg    = Config()

entities, entity_index, train_queries, test_queries, held_out_names = \
    load_dataset("dataset.json")

tokenizer         = SROTokenizer.from_dataset("dataset.json")
cfg.vocab_size    = tokenizer.vocab_size

cfg.d_model       = 256
cfg.d_semantic    = 192
cfg.d_positional  = 64
cfg.max_seq_len   = 128

valid_answers_map = build_valid_answers(entities)

summed_model = load_model("outputs/summed/summed_model_best2.pt",
                          "summed", cfg, device)
disent_model = load_model("outputs/disentangled/concatenated_model_best2.pt",
                          "disentangled", cfg, device)

@torch.no_grad()
def show_examples(model, model_name, queries, n_extraction=8, n_composition=8):
    model.eval()
    print(f"\n{'═'*65}")
    print(f" {model_name}")
    print(f"{'═'*65}")

    print(f"\n── Extraction queries (single correct answer) ──")
    ext_qs = [q for q in queries if q.query_type == "extraction"][:n_extraction]
    ext_correct = 0
    for q in ext_qs:
        input_ids = tokenizer.encode(q.prompt, return_tensors="pt").to(device)
        logits, _ = model(input_ids)
        probs     = F.softmax(logits[0, -1, :], dim=-1)
        top3_ids  = probs.argsort(descending=True)[:3].tolist()
        top3      = [(tokenizer.convert_ids_to_tokens(i), round(probs[i].item(), 3))
                     for i in top3_ids]
        ans_tok   = tokenizer.encode(q.answer)[0]
        correct   = probs.argmax().item() == ans_tok
        ext_correct += int(correct)
        status    = "✓" if correct else "✗"
        print(f"  {status} {q.subject} → {q.relation}")
        print(f"     Expected : {q.answer}")
        print(f"     Top 3    : {top3[0][0]}({top3[0][1]})  "
              f"{top3[1][0]}({top3[1][1]})  {top3[2][0]}({top3[2][1]})")
    print(f"  Extraction accuracy: {ext_correct}/{n_extraction}")

    print(f"\n── Composition queries (any entity sharing the attribute is correct) ──")
    comp_qs = [q for q in queries if q.query_type == "composition"][:n_composition]
    comp_correct = 0
    for q in comp_qs:
        valid     = valid_answers_map.get((q.subject, q.relation), set())
        input_ids = tokenizer.encode(q.prompt, return_tensors="pt").to(device)
        logits, _ = model(input_ids)
        probs     = F.softmax(logits[0, -1, :], dim=-1)
        top3_ids  = probs.argsort(descending=True)[:3].tolist()
        top3      = [(tokenizer.convert_ids_to_tokens(i), round(probs[i].item(), 3))
                     for i in top3_ids]
        top_word  = top3[0][0]
        correct   = top_word in valid
        comp_correct += int(correct)
        status    = "✓" if correct else "✗"

        # Show the attribute value being matched
        attr      = q.relation.replace("same_", "").replace("_as", "")
        attr_val  = entity_index[q.subject].get(attr, "?")

        print(f"  {status} {q.subject} → {q.relation}")
        print(f"     Subject's {attr}: {attr_val}  |  Valid answers: {len(valid)} entities")
        print(f"     Predicted: {top3[0][0]}({top3[0][1]})  "
              f"{top3[1][0]}({top3[1][1]})  {top3[2][0]}({top3[2][1]})")
        print(f"     Top prediction in valid set: {correct}")
    print(f"  Composition accuracy: {comp_correct}/{n_composition}")

    # Held-out composition examples
    print(f"\n── Held-out composition (generalization — never seen during training) ──")
    held_qs = [q for q in queries
               if q.query_type == "composition"
               and q.subject in held_out_names][:n_composition]
    held_correct = 0
    for q in held_qs:
        valid     = valid_answers_map.get((q.subject, q.relation), set())
        input_ids = tokenizer.encode(q.prompt, return_tensors="pt").to(device)
        logits, _ = model(input_ids)
        probs     = F.softmax(logits[0, -1, :], dim=-1)
        top3_ids  = probs.argsort(descending=True)[:3].tolist()
        top3      = [(tokenizer.convert_ids_to_tokens(i), round(probs[i].item(), 3))
                     for i in top3_ids]
        top_word  = top3[0][0]
        correct   = top_word in valid
        held_correct += int(correct)
        status    = "✓" if correct else "✗"

        attr     = q.relation.replace("same_", "").replace("_as", "")
        attr_val = entity_index[q.subject].get(attr, "?")

        print(f"  {status} {q.subject} → {q.relation}")
        print(f"     Subject's {attr}: {attr_val}  |  Valid answers: {len(valid)} entities")
        print(f"     Predicted: {top3[0][0]}({top3[0][1]})  "
              f"{top3[1][0]}({top3[1][1]})  {top3[2][0]}({top3[2][1]})")
        print(f"     Top prediction in valid set: {correct}")
    print(f"  Held-out composition accuracy: {held_correct}/{n_composition}")


show_examples(summed_model, "SUMMED (baseline)",       test_queries)
show_examples(disent_model, "DISENTANGLED (method)",   test_queries)

  Loaded 1000 entity names from dataset.json
SROTokenizer built:
  Special tokens:   9
  Attribute values: 35
  Relation words:   8
  Entity names:     1000  (from dataset.json)
  Total vocab:      1052
  [summed] 3,730,944 parameters  | 4 blocks | 4 heads
  [disentangled] 3,639,040 parameters  | 4 blocks | 4 heads

═════════════════════════════════════════════════════════════════
 SUMMED (baseline)
═════════════════════════════════════════════════════════════════

── Extraction queries (single correct answer) ──
  ✓ Bliblax-210 → class
     Expected : warrior
     Top 3    : warrior(0.889)  builder(0.099)  scholar(0.005)
  ✓ Bliblax-210 → color
     Expected : green
     Top 3    : green(0.464)  orange(0.204)  yellow(0.167)
  ✗ Bliblax-210 → material
     Expected : wood
     Top 3    : stone(0.615)  glass(0.329)  plastic(0.046)
  ✓ Bliblax-210 → origin
     Expected : region_B
     Top 3    : region_B(0.999)  region_C(0.001)  region_E(0.0)
  ✓ Bliblax-210 → shape
     Expected : star